In [ ]:
# =====================================================
#  Dataset Builder for G-code Layers (with Areas)
# =====================================================
# Features:
# - Reads multiple .gcode files
# - Extracts layers dynamically by Z height
# - Skips the 3rd layer (index 2)
# - Pads to maximum number of layers & points
# - Stores per-layer convex hull area (raw + normalized)
# - Outputs a clean .npy dataset
# =====================================================


#**** RUN THE LAST CELL 1st then continue 

🧾 Found 50 G-code files.
📏 Max layers (after skipping 3rd): 199, Max points per layer: 2740
Label distribution: Counter({'cube': 5, 'cylinder': 5, 'flange': 5, 'gear_bevel': 5, 'gear_spur': 5, 'lead_screw': 5, 'nut': 5, 'propeller': 5, 'pyramid': 5, 'sphere': 5})
Label map: {'cube': 0, 'cylinder': 1, 'flange': 2, 'gear_bevel': 3, 'gear_spur': 4, 'lead_screw': 5, 'nut': 6, 'propeller': 7, 'pyramid': 8, 'sphere': 9}
✅ Saved: F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1\gcode_layers_V1.npy
📐 Data shape: (50,)
🏷️ Labels: ['cube', 'cube', 'cube', 'cube', 'cube', 'cylinder', 'cylinder', 'cylinder', 'cylinder', 'cylinder', 'flange', 'flange', 'flange', 'flange', 'flange', 'gear_bevel', 'gear_bevel', 'gear_bevel', 'gear_bevel', 'gear_bevel', 'gear_spur', 'gear_spur', 'gear_spur', 'gear_spur', 'gear_spur', 'lead_screw', 'lead_screw', 'lead_screw', 'lead_screw', 'lead_screw', 'nut', 'nut', 'nut', 'nut', 'nut', 'propeller', 'propeller', 'propeller', 'propeller', 'propeller', 'pyramid', '

In [8]:
import numpy as np
from collections import defaultdict

data = np.load(
    r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1\gcode_layers_V1.npy",
    allow_pickle=True
).item()

layers_per_file = data["data"]        # list of accepted layers per file
label_names     = data["label_names"] # label per file

total_layers_per_label = defaultdict(int)

for lbl, layers in zip(label_names, layers_per_file):
    total_layers_per_label[lbl] += len(layers)

print("📊 TOTAL ACCEPTED LAYERS PER LABEL (across ALL files)")
print("-" * 55)
for lbl in sorted(total_layers_per_label):
    print(f"{lbl:12s} : {total_layers_per_label[lbl]}")

print("\n✅ Grand total layers:",
      sum(total_layers_per_label.values()))


📊 TOTAL ACCEPTED LAYERS PER LABEL (across ALL files)
-------------------------------------------------------
cube         : 590
cylinder     : 590
flange       : 590
gear_bevel   : 590
gear_spur    : 590
lead_screw   : 590
nut          : 590
propeller    : 590
pyramid      : 590
sphere       : 590

✅ Grand total layers: 5900


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Load dataset ===
npy_path = r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1\gcode_layers_V1.npy"
loaded = np.load(npy_path, allow_pickle=True).item()

X = loaded["data"]
y = loaded["labels"]
label_map = loaded["label_map"]
name_map = {v: k for k, v in label_map.items()}

# === Collect layers with labels and local layer index ===
layers = []
for i, file in enumerate(X):
    shape_name = name_map[y[i]]
    local_layer_id = 1
    for layer in file:
        if np.any(layer):  
            points = layer[~np.all(layer == 0, axis=1)]
            if len(points) < 3:  # Skip "dot" or invalid layers
                continue
            layers.append((points, shape_name, local_layer_id))
            local_layer_id += 1

print(f"✅ Total valid layers to plot: {len(layers)}")

# === Plotting config ===
cols = 5
rows_per_fig = 4
plots_per_fig = cols * rows_per_fig

for start in range(0, len(layers), plots_per_fig):
    fig, axs = plt.subplots(rows_per_fig, cols, figsize=(5 * cols, 5 * rows_per_fig))
    axs = axs.flatten()

    for i in range(plots_per_fig):
        idx = start + i
        if idx >= len(layers):
            break

        points, shape_name, local_id = layers[idx]

        axs[i].scatter(points[:, 0], points[:, 1], s=10, color='blue')
        axs[i].set_title(f"{shape_name} (Layer {local_id})")
        #axs[i].set_xlim(-30, 140)
        #axs[i].set_ylim(-30, 140)
        axs[i].set_aspect('equal')
        axs[i].grid(True)

    for j in range(i + 1, plots_per_fig):
        axs[j].axis('off')

    plt.tight_layout()
    plt.show()


In [5]:
import re
import numpy as np
from pathlib import Path

# ================================
# CONFIG
# ================================
gcode_dir = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1")
outlier_threshold = 1e-3

gcode_files = sorted(gcode_dir.glob("*.gcode"))
print(f"Scanning {len(gcode_files)} files...\n")

# ================================
# G-code layer extractor
# ================================
def extract_layers_dynamically(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    layer_dict = {}
    current_z = None
    last_x, last_y = None, None

    for line in lines:
        if line.startswith(";") or not any(axis in line for axis in ["X", "Y", "Z"]):
            continue

        x_match = re.search(r"X(-?\d*\.?\d+)", line)
        y_match = re.search(r"Y(-?\d*\.?\d+)", line)
        z_match = re.search(r"Z(-?\d*\.?\d+)", line)

        if z_match:
            current_z = round(float(z_match.group(1)), 3)
            if current_z not in layer_dict:
                layer_dict[current_z] = []

        if x_match:
            last_x = float(x_match.group(1))
        if y_match:
            last_y = float(y_match.group(1))

        if current_z is not None and last_x is not None and last_y is not None:
            if abs(last_x) > outlier_threshold or abs(last_y) > outlier_threshold:
                layer_dict[current_z].append([last_x, last_y])

    sorted_z = sorted(layer_dict.keys())
    return [np.array(layer_dict[z]) for z in sorted_z]

# ================================
# FULL SCAN (NO EARLY EXIT)
# ================================
issues = []  # store all problems here

for file_idx, file in enumerate(gcode_files):
    layers = extract_layers_dynamically(file)

    for layer_idx, layer in enumerate(layers):

        # non-array
        if not isinstance(layer, np.ndarray):
            issues.append({
                "file": file.name,
                "layer": layer_idx,
                "issue": "non-numpy",
                "shape": None
            })
            continue

        # malformed shape
        if layer.ndim != 2 or layer.shape[1] != 2:
            issues.append({
                "file": file.name,
                "layer": layer_idx,
                "issue": "malformed-shape",
                "shape": layer.shape
            })
            continue

        # empty layer
        if layer.shape[0] == 0:
            issues.append({
                "file": file.name,
                "layer": layer_idx,
                "issue": "empty-layer",
                "shape": layer.shape
            })
            continue

# ================================
# REPORT
# ================================
if len(issues) == 0:
    print("✅ Scan complete: NO malformed layers found.")
else:
    print(f"\n🚨 FOUND {len(issues)} PROBLEMATIC LAYERS 🚨\n")
    for i, issue in enumerate(issues, 1):
        print(
            f"{i:02d}. File: {issue['file']}, "
            f"Layer: {issue['layer']}, "
            f"Issue: {issue['issue']}, "
            f"Shape: {issue['shape']}"
        )

    # summary per file
    print("\n📊 SUMMARY PER FILE:")
    from collections import Counter
    file_counts = Counter(i["file"] for i in issues)
    for f, c in file_counts.items():
        print(f"  {f}: {c} problematic layers")


Scanning 50 files...


🚨 FOUND 50 PROBLEMATIC LAYERS 🚨

01. File: cube_02.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
02. File: cube_04.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
03. File: cube_06.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
04. File: cube_08.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
05. File: cube_10.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
06. File: cylinder_02.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
07. File: cylinder_04.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
08. File: cylinder_06.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
09. File: cylinder_08.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
10. File: cylinder_10.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
11. File: flange_02.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
12. File: flange_04.gcode, Layer: 2, Issue: malformed-shape, Shape: (0,)
13. File: flange_06.gcode, Layer: 2, Issue: malformed-shape, Shape: 

In [6]:
import re
import numpy as np
from pathlib import Path
from collections import Counter

gcode_dir = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1")
outlier_threshold = 1e-3

def extract_layers_dynamically(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    layer_dict = {}
    current_z = None
    last_x, last_y = None, None

    for line in lines:
        if line.startswith(";") or not any(axis in line for axis in ["X", "Y", "Z"]):
            continue

        x_match = re.search(r"X(-?\d*\.?\d+)", line)
        y_match = re.search(r"Y(-?\d*\.?\d+)", line)
        z_match = re.search(r"Z(-?\d*\.?\d+)", line)

        if z_match:
            current_z = round(float(z_match.group(1)), 3)
            if current_z not in layer_dict:
                layer_dict[current_z] = []

        if x_match:
            last_x = float(x_match.group(1))
        if y_match:
            last_y = float(y_match.group(1))

        if current_z is not None and last_x is not None and last_y is not None:
            if abs(last_x) > outlier_threshold or abs(last_y) > outlier_threshold:
                layer_dict[current_z].append([last_x, last_y])

    return [np.array(layer_dict[z]) for z in sorted(layer_dict.keys())]


# ================================
# COLLECT BAD LAYER INDICES
# ================================
bad_layer_indices = []

for file in gcode_dir.glob("*.gcode"):
    layers = extract_layers_dynamically(file)

    for idx, layer in enumerate(layers):
        if layer.ndim != 2 or layer.shape[0] == 0 or layer.shape[1] != 2:
            bad_layer_indices.append(idx)

# ================================
# REPORT
# ================================
print("Bad layer index frequency:")
print(Counter(bad_layer_indices))


Bad layer index frequency:
Counter({2: 47, 1: 3})


In [7]:
# =====================================================
#  Dataset Builder for G-code Layers (with Areas)
# =====================================================
# Features:
# - Reads multiple .gcode files
# - Extracts layers dynamically by Z height
# - Skips the 3rd layer (index 2)
# - Pads to maximum number of layers & points
# - Stores per-layer convex hull area (raw + normalized)
# - Outputs a clean .npy dataset
# =====================================================

import os
import re
import numpy as np
from pathlib import Path


# =====================================================
#  Configuration
# =====================================================
gcode_dir = Path(r"F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1")

# Find all G-code files
gcode_files = list(gcode_dir.glob("*.gcode"))
print(f"🧾 Found {len(gcode_files)} G-code files.")

# Parameters
outlier_threshold = 1e-3 # filter near-zero points like (0,0)

def normalize_and_resample_layer(pc, target_size):
    pc = np.asarray(pc, dtype=np.float32)

    if pc.ndim != 2 or pc.shape[0] == 0:
        return np.zeros((target_size, 2), dtype=np.float32)

    # center
    centroid = np.mean(pc, axis=0)
    pc = pc - centroid

    # box normalize to [-1, 1]
    min_xy = pc.min(axis=0)
    max_xy = pc.max(axis=0)
    scale = np.maximum(max_xy - min_xy, 1e-8)
    pc = 2.0 * (pc - min_xy) / scale - 1.0

    # resample
    if pc.shape[0] < target_size:
        rep = (target_size // pc.shape[0]) + 1
        pc = np.tile(pc, (rep, 1))[:target_size]
    elif pc.shape[0] > target_size:
        pc = pc[:target_size]

    return pc
# =====================================================
#  Geometry Helper Functions
# =====================================================
def convex_hull(points: np.ndarray):
    """Compute convex hull vertices (CCW) using monotone chain algorithm."""
    pts = np.unique(points, axis=0)
    if len(pts) <= 1:
        return pts

    # Sort by x, then y
    pts = pts[np.lexsort((pts[:,1], pts[:,0]))]

    def cross(o, a, b):
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])

    # Build lower hull
    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(tuple(p))

    # Build upper hull
    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(tuple(p))

    hull = np.array(lower[:-1] + upper[:-1], dtype=float)
    return hull

def deduplicate_layers_by_label_area(layers, areas_norm, label_name):
    """
    layers: list of np.ndarray (raw layers, variable points)
    areas_norm: list or array of normalized areas (same length as layers)
    label_name: string label (e.g., 'cube')
    """
    best = {}  # key=(label, area_norm) -> (layer, num_points)

    for layer, area in zip(layers, areas_norm):
        key = (label_name, round(float(area), 6))  # round to avoid float noise
        npts = layer.shape[0]

        if key not in best or npts > best[key][1]:
            best[key] = (layer, npts)

    return [v[0] for v in best.values()]

def polygon_area(points: np.ndarray) -> float:
    """Shoelace formula for polygon area. Points must be ordered."""
    if len(points) < 3:
        return 0.0
    x, y = points[:,0], points[:,1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def extract_base_label(filepath: Path):
    """
    Extracts shape class from filename.
    Examples:
        cube_02.gcode        -> cube
        gear_spur_10.gcode  -> gear_spur
        flange_04.gcode     -> flange
    """
    name = filepath.stem              # e.g. "gear_spur_10"
    return "_".join(name.split("_")[:-1])
    
def convex_hull_area(points: np.ndarray) -> float:
    """Compute convex hull area for a set of points."""
    if len(points) < 3:
        return 0.0
    hull = convex_hull(points)
    return polygon_area(hull)
def keep_half_layers(layers):
    if len(layers) == 0:
        return layers
    return layers[: len(layers) // 2]
def duplicate_layers_to_target(layers, target=590):
    if len(layers) == 0:
        return layers
    out = []
    i = 0
    while len(out) < target:
        out.append(layers[i % len(layers)])
        i += 1
    return out
# =====================================================
#  G-code Parsing
# =====================================================
def extract_layers_dynamically(filepath):
    """Parse one G-code file and extract layers as arrays of (x,y) points."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    layer_dict = {}
    current_z = None
    last_x, last_y = None, None

    for line in lines:
        if line.startswith(";") or not any(axis in line for axis in ["X", "Y", "Z"]):
            continue

        # Match XYZ coordinates
        x_match = re.search(r"X(-?\d*\.?\d+)", line)
        y_match = re.search(r"Y(-?\d*\.?\d+)", line)
        z_match = re.search(r"Z(-?\d*\.?\d+)", line)

        if z_match:
            current_z = round(float(z_match.group(1)), 3)
            if current_z not in layer_dict:
                layer_dict[current_z] = []

        if x_match:
            last_x = float(x_match.group(1))
        if y_match:
            last_y = float(y_match.group(1))

        # Save point if valid
        if current_z is not None and last_x is not None and last_y is not None:
            if abs(last_x) > outlier_threshold or abs(last_y) > outlier_threshold:
                layer_dict[current_z].append([last_x, last_y])

    # Sort by Z and convert to arrays
    sorted_z = sorted(layer_dict.keys())
    sorted_layers = [np.array(layer_dict[z]) for z in sorted_z]

    return sorted_layers


# =====================================================
#  Extract Dataset from All Files
# =====================================================
all_data = []
labels = []

max_layers = 0
max_points = 0

for file in gcode_files:
    layers = extract_layers_dynamically(file)
    layers = [layer for i, layer in enumerate(layers) if i not in (0,1,2)]
    
    # filter invalid layers first
    layers = [
        layer for layer in layers
        if layer.shape[0] >= 10 and convex_hull_area(layer) > 0.0
    ]
    
    # compute areas (raw, before normalization)
    areas_raw_tmp = [convex_hull_area(l) for l in layers]
    
    # normalize areas TEMPORARILY for dedup key
    if len(areas_raw_tmp) > 0:
        a_min, a_max = min(areas_raw_tmp), max(areas_raw_tmp)
        if a_max - a_min < 1e-8:
            areas_norm_tmp = [0.0 for _ in areas_raw_tmp]
        else:
            areas_norm_tmp = [(a - a_min) / (a_max - a_min) for a in areas_raw_tmp]
    else:
        areas_norm_tmp = []
    


    all_data.append(layers)
    base_label = extract_base_label(file)   # ✅ ADD
    labels.append(base_label)               # ✅ CHANGE

    max_layers = max(max_layers, len(layers))
    max_points = max(max_points, max((len(l) for l in layers), default=0))

print(f"📏 Max layers (after skipping 3rd): {max_layers}, Max points per layer: {max_points}")

from collections import defaultdict

TARGET_LAYERS = 590

label_pool = defaultdict(list)

# collect ALL layers per label (across files)
for layers, label in zip(all_data, labels):
    for layer in layers:
        label_pool[label].append(layer)

# deduplicate by (label, area) keeping max-point layer
final_all_data = []
final_labels = []

for label, layers in label_pool.items():
    area_map = {}

    for layer in layers:
        area = round(convex_hull_area(layer), 6)
        if area not in area_map or layer.shape[0] > area_map[area].shape[0]:
            area_map[area] = layer

    unique_layers = list(area_map.values())

    # sequential duplication to 590
    duplicated = []
    i = 0
    while len(duplicated) < TARGET_LAYERS:
        duplicated.append(unique_layers[i % len(unique_layers)])
        i += 1

    final_all_data.append(duplicated)
    final_labels.append(label)


# =====================================================
final_array = []
layer_index_array = []
areas_array = []

for layers in final_all_data:
    padded_layers = []
    layer_indices = []
    layer_areas = []

    for i, layer in enumerate(layers):

        # --- area before padding ---
        area_val = convex_hull_area(layer) if layer.shape[0] > 2 else 0.0
        layer_areas.append(area_val)

        # --- normalize, center, resample layer ---
        layer_norm = normalize_and_resample_layer(layer, max_points)
        padded_layers.append(layer_norm)
        layer_indices.append(i)   # ← ADD THIS LINE

    # Pad extra empty layers

    final_array.append(padded_layers)
    layer_index_array.append(layer_indices)
    areas_array.append(layer_areas)

final_array = np.array(final_array, dtype=object)
layer_index_array = np.array(layer_index_array, dtype=object)
areas_array = np.array(areas_array, dtype=object)          # [num_files, max_layers]
# =====================================================
#  Labels   ✅ MOVED HERE
# =====================================================
labels = final_labels

unique_labels = sorted(set(labels))
label_to_idx = {name: i for i, name in enumerate(unique_labels)}
int_labels = np.array([label_to_idx[l] for l in labels])

from collections import Counter
print("Label distribution:", Counter(labels))
print("Label map:", label_to_idx)

# =====================================================
#  Per-label Area Normalization (RAW → RAWV2 → NORM)
# =====================================================
RAWV2_MAX = 400.0

areas_rawv2 = np.array([np.zeros(len(a)) for a in areas_array], dtype=object)

for label in unique_labels:
    file_idxs = [i for i, l in enumerate(labels) if l == label]

    label_areas = np.concatenate([areas_array[i] for i in file_idxs])

    label_min = label_areas.min()
    label_max = label_areas.max()

    if label_max - label_min < 1e-8:
        continue

    for i in file_idxs:
        areas_rawv2[i] = (
            (areas_array[i] - label_min) / (label_max - label_min)
        ) * RAWV2_MAX
    
# Final normalized area (0–1)
areas_norm = areas_rawv2 / RAWV2_MAX


# =====================================================
#  Save Dataset
# =====================================================
dataset = {
    "data": final_array,               # (N, L, P, 2)
    "labels": int_labels,              # (N,)
    "label_names": labels,             # e.g., ["cube", "gear-spur", ...]
    "label_map": label_to_idx,         # str -> int
    "layer_indices": layer_index_array,# (N, L)
    "areas_raw": areas_array,          # original raw
    "areas_rawv2": areas_rawv2,        # per-label scaled 0–400
    "areas_norm": areas_norm        # final normalized 0–1
}

output_path = gcode_dir / "gcode_layers_V1.npy"
np.save(output_path, dataset, allow_pickle=True)

print(f"✅ Saved: {output_path}")
print(f"📐 Data shape: {final_array.shape}")
print(f"🏷️ Labels: {labels}")
print(f"🔑 Label map: {label_to_idx}")
print(f"🔢 Layer indices shape: {layer_index_array.shape}")
print(
    f"📏 Raw Areas  | shape={areas_array.shape}, "
    f"range=({np.min(np.concatenate(areas_array)):.3f}, {np.max(np.concatenate(areas_array)):.3f})"
)

print(
    f"📏 RawV2 Areas| shape={areas_rawv2.shape}, "
    f"range=({np.min(np.concatenate(areas_rawv2)):.3f}, {np.max(np.concatenate(areas_rawv2)):.3f})"
)

print(
    f"📏 Norm Areas | shape={areas_norm.shape}, "
    f"range=({np.min(np.concatenate(areas_norm)):.3f}, {np.max(np.concatenate(areas_norm)):.3f})"
)

🧾 Found 50 G-code files.
📏 Max layers (after skipping 3rd): 398, Max points per layer: 2900
Label distribution: Counter({'cube': 1, 'cylinder': 1, 'flange': 1, 'gear_bevel': 1, 'gear_spur': 1, 'lead_screw': 1, 'nut': 1, 'propeller': 1, 'pyramid': 1, 'sphere': 1})
Label map: {'cube': 0, 'cylinder': 1, 'flange': 2, 'gear_bevel': 3, 'gear_spur': 4, 'lead_screw': 5, 'nut': 6, 'propeller': 7, 'pyramid': 8, 'sphere': 9}
✅ Saved: F:\JupyterWork\Testing GAN\CGAN V2 for AMF\DatasetRV1\gcode_layers_V1.npy
📐 Data shape: (10, 590, 2900, 2)
🏷️ Labels: ['cube', 'cylinder', 'flange', 'gear_bevel', 'gear_spur', 'lead_screw', 'nut', 'propeller', 'pyramid', 'sphere']
🔑 Label map: {'cube': 0, 'cylinder': 1, 'flange': 2, 'gear_bevel': 3, 'gear_spur': 4, 'lead_screw': 5, 'nut': 6, 'propeller': 7, 'pyramid': 8, 'sphere': 9}
🔢 Layer indices shape: (10, 590)
📏 Raw Areas  | shape=(10, 590), range=(1.004, 754.914)
📏 RawV2 Areas| shape=(10, 590), range=(0.000, 400.000)
📏 Norm Areas | shape=(10, 590), range=(0.00